<a href="https://colab.research.google.com/github/almin-byte/million-headlines-topic-modeling/blob/main/Topic_Modeling_sklearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [72]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
# import packages

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn import decomposition
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.model_selection import train_test_split

In [43]:
# download tokenizer

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [44]:
# load data

df = pd.read_csv('/content/abcnews-date-text.csv')

In [45]:
df

,publish_date,headline_text
0,20030219,aba decides against community broadcasting licence
1,20030219,act fire witnesses must be aware of defamation
2,20030219,a g calls for infrastructure protection summit
3,20030219,air nz staff in aust strike for pay rise
4,20030219,air nz strike to affect australian travellers
...,...,...
1244179,20211231,two aged care residents die as state records 2;093
1244180,20211231,victoria records 5;919 new cases and seven deaths
1244181,20211231,wa delays adopting new close contact definition
1244182,20211231,western ringtail possums found badly dehydrated in heatwave


In [46]:
# isolate the headlines

headlines_df = df.get(['headline_text'])

In [47]:
pd.set_option('display.max_colwidth', None)
headlines_df

,headline_text
0,aba decides against community broadcasting licence
1,act fire witnesses must be aware of defamation
2,a g calls for infrastructure protection summit
3,air nz staff in aust strike for pay rise
4,air nz strike to affect australian travellers
...,...
1244179,two aged care residents die as state records 2;093
1244180,victoria records 5;919 new cases and seven deaths
1244181,wa delays adopting new close contact definition
1244182,western ringtail possums found badly dehydrated in heatwave


In [48]:
# split the dataset (keep ~50,000 headlines)

X_train, X_hold = train_test_split(headlines_df, test_size = .959, random_state=111)

In [49]:
X_train

,headline_text
794191,wilcannia fun day a positive sign for tourism
1125387,departing graham arnold says sky blues best ever a league side
905700,queensland firefighters face sack over alleged sex poll
1198396,pencil pines in tasmania seeding botanist coronavirus
247516,nz coast to victory in top end
...,...
942761,david lynch confirms he wont direct twin peaks
102486,rain delays gymkhana
135892,grampians blaze still burning
534484,track plan angers 4wd fans and conservationists


In [50]:
# import stemmer

stemmer = PorterStemmer()

In [51]:
# create a function to tokenize the data using the nltk word tokenizer
# filter any words less than 3 chars

def tokenize(text):
  tokens = [word for word in nltk.word_tokenize(text) if len(word) > 3]
  stems = [stemmer.stem(item) for item in tokens]
  return tokens

In [52]:
# initialize the TfidfVectorizer (text data --> vector format)
# use_idf = False --> only tone frequency is executed
# norm = None --> appear as a count vectorizer
# max_df = .75 --> word appears in maximum 75% of documents
# min_df = 50 --> word needs to be in at least 50 documents
# passing headlines_text column in training df to fit_transform method of vectorizer
# yields output feature vector

vectorizer_tf = TfidfVectorizer(tokenizer=tokenize, stop_words='english', max_df=.75, min_df=50, max_features=10000, use_idf=False, norm=None)
tf_vectors = vectorizer_tf.fit_transform(X_train.headline_text)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [53]:
vectorizer_tf.get_feature_names_out()

array(['2013', '2014', '2015', '2016', 'abbott', 'aboriginal', 'abuse',
       'accc', 'access', 'accident', 'accused', 'action', 'address',
       'adelaide', 'admits', 'afghan', 'afghanistan', 'africa', 'aged',
       'agreement', 'ahead', 'aims', 'airport', 'alcohol', 'alice',
       'allegations', 'alleged', 'amid', 'analysis', 'andrew', 'animal',
       'anniversary', 'announces', 'anti', 'anzac', 'apologises',
       'appeal', 'armed', 'army', 'arrest', 'arrested', 'arrests',
       'asbestos', 'ashes', 'asia', 'asian', 'assault', 'asylum',
       'attack', 'attacks', 'august', 'aussie', 'aussies', 'aust',
       'australia', 'australian', 'australians', 'australias',
       'authorities', 'award', 'awards', 'away', 'baby', 'backs', 'bail',
       'bali', 'bank', 'banks', 'bans', 'base', 'bashing', 'battle',
       'beach', 'beat', 'beef', 'begin', 'begins', 'bendigo', 'best',
       'better', 'biggest', 'bikie', 'billion', 'bird', 'black', 'blamed',
       'blast', 'blaze', 'blu

In [54]:
# lda model

lda = decomposition.LatentDirichletAllocation(n_components=10, max_iter=3, learning_method='online', learning_offset=50, n_jobs=-1, random_state=111)

W1 = lda.fit_transform(tf_vectors)
H1 = lda.components_

In [55]:
W1

array([[0.025     , 0.025     , 0.025     , ..., 0.025     , 0.025     ,
        0.025     ],
       [0.02      , 0.02      , 0.02      , ..., 0.22      , 0.02      ,
        0.02      ],
       [0.01666667, 0.01666667, 0.01666667, ..., 0.01666667, 0.01666667,
        0.01666667],
       ...,
       [0.05      , 0.05      , 0.05      , ..., 0.55      , 0.05      ,
        0.05      ],
       [0.025     , 0.27499998, 0.025     , ..., 0.025     , 0.025     ,
        0.27500013],
       [0.025     , 0.025     , 0.025     , ..., 0.27499997, 0.025     ,
        0.025     ]])

In [56]:
# prints out the top 5 words contributing to each topic

num_words=5

vocab = np.array(vectorizer_tf.get_feature_names_out())

top_words = lambda t: [vocab[i] for i in np.argsort(t)[:-num_words-1:-1]]
topic_words = ([top_words(t) for t in H1])
topics = [' '.join(t) for t in topic_words]

In [57]:
topics

['report china labor country rain',
 'crash charged home attack trial',
 'police woman missing melbourne indigenous',
 'australia interview murder coronavirus workers',
 'face queensland test charges group',
 'health government year hospital killed',
 'govt water accused election urged',
 'says sydney world south rural',
 'council australian north boost power',
 'court plan calls coast death']

In [62]:
# display the topic distribution of each document
# determine the dominant topic of each document
# display the final topic decisions of the model

colnames = ["Topic" + str(i) for i in range(lda.n_components)]
docnames = ["Doc" + str(i) for i in range(len(X_train.headline_text))]
df_doc_topic = pd.DataFrame(np.round(W1, 2), columns=colnames, index=docnames)
significant_topic = np.argmax(df_doc_topic.values, axis=1)
df_doc_topic['dominant_topic'] = significant_topic

In [63]:
df_doc_topic

,Topic0,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,dominant_topic
Doc0,0.03,0.03,0.03,0.03,0.03,0.03,0.77,0.03,0.03,0.03,6
Doc1,0.02,0.02,0.02,0.02,0.62,0.02,0.02,0.22,0.02,0.02,4
Doc2,0.02,0.02,0.02,0.02,0.85,0.02,0.02,0.02,0.02,0.02,4
Doc3,0.03,0.03,0.03,0.70,0.03,0.03,0.03,0.03,0.03,0.03,3
Doc4,0.03,0.03,0.03,0.03,0.03,0.03,0.37,0.03,0.03,0.37,6
...,...,...,...,...,...,...,...,...,...,...,...
Doc51006,0.53,0.03,0.03,0.03,0.03,0.03,0.03,0.27,0.03,0.03,0
Doc51007,0.37,0.03,0.37,0.03,0.03,0.03,0.03,0.03,0.03,0.03,0
Doc51008,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.55,0.05,0.05,7
Doc51009,0.03,0.27,0.03,0.03,0.27,0.03,0.03,0.03,0.03,0.28,9


In [64]:
X_train.head()

,headline_text
794191,wilcannia fun day a positive sign for tourism
1125387,departing graham arnold says sky blues best ever a league side
905700,queensland firefighters face sack over alleged sex poll
1198396,pencil pines in tasmania seeding botanist coronavirus
247516,nz coast to victory in top end


In [65]:
df_doc_topic.head()

,Topic0,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,dominant_topic
Doc0,0.03,0.03,0.03,0.03,0.03,0.03,0.77,0.03,0.03,0.03,6
Doc1,0.02,0.02,0.02,0.02,0.62,0.02,0.02,0.22,0.02,0.02,4
Doc2,0.02,0.02,0.02,0.02,0.85,0.02,0.02,0.02,0.02,0.02,4
Doc3,0.03,0.03,0.03,0.70,0.03,0.03,0.03,0.03,0.03,0.03,3
Doc4,0.03,0.03,0.03,0.03,0.03,0.03,0.37,0.03,0.03,0.37,6
